In [1]:
%%capture
%pip install evaluate

In [3]:
# Les importations
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback
from datasets import load_dataset,Dataset, DatasetDict
import evaluate
import numpy as np
from torch.cuda.amp import autocast, GradScaler
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

/home/koubra/Documents/Projet_IA/Chadian arabic update/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1791301459.279856    6232 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791301471.233357    6232 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Chargement du dataset

In [4]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [13]:
data_arb = dataset_loader("../Data/full_dataset/arabe_tchad3.txt")
data_fr = dataset_loader("../Data/full_dataset/V_francais_3.txt")

In [14]:
print(type(data_arb))

<class 'list'>


In [15]:
# for trying 
data_arb = data_arb[:3000] 
data_fr = data_fr[:3000]

### Diviosion du dataset en train, test et validation

In [ ]:
# Création du dictionnaire
dataset_dict = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb, data_fr)]}

# Création du dataset
dataset = Dataset.from_dict(dataset_dict)
# On va diviser notre dataset en deux parties
 #train_dataset =  DatasetDict({"train":test['train']})

train_test_dataset = dataset.train_test_split(test_size=0.2, seed= 42, shuffle= True)

test_val_dataset = train_test_dataset['test'].train_test_split(test_size= 0.5, seed= 42)

train_test_val_dataset = DatasetDict ({
    'train' : train_test_dataset['train'],
    'val'   : test_val_dataset['train'],
    'test'  : test_val_dataset['test']
})
train_test_val_dataset.save_to_disk("../Data/train_test_split")


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 8786.64 examples/s] 
